# DRVI Dataset Curation

Produces a cleaned, re-annotated copy of the dataset from the candidates in Notebook C (Tregs/pDC/Plasma cells newly annotated; B/NK doublets, B/myeloid doublets, the IGHG3+/k28 mini-cluster, suspected PF4+ platelets/ambient RNA (megakaryocytes excluded), and Leiden cluster 13 removed) and validates the result (cell-type UMAP, B-cell subclustering check, overall Leiden UMAP).

The immunoglobulin gene analysis (IGH*/IGL* across timepoints) is in its own notebook: `2_E_drvi_ig_gene_timepoint_analysis.ipynb`.

## 1. Create annotated & cleaned copy

*Source: `2_8_a_drvi_create_annotated_copy.py`*

Creates an annotated/cleaned copy of
data_for_practicum_post_integration.h5ad, based on the DRVI factor
threshold candidates from 2_7_a/2_7_c:

Newly annotated (new column `cell_type_curated`, original `cell_type_Scanorama`
kept unchanged):
  - DR9-  -> "Tregs"
  - DR17+ -> "pDC"
  - DR28+ -> "Plasma cells"
  (DR4+ / platelets deliberately NOT annotated — instead removed via PF4
   gene expression, see below.)

Removed (suspected doublets/batch artifacts):
  - DR21+ candidates within B cells (NK signal in B cells ->
    suspected B/NK doublets)
  - DR35+ candidates within Monocytes-CD14/Dendritic/Monocytes-CD16_FCGR3A
    (B-cell signal in myeloid cells -> suspected B/myeloid doublets)
  - All cells from candidate_cells_gene_IGHG3_in_Bcell.csv (IGHG3+ B cells,
    almost exclusively driven by one donor (k28) -> suspected
    batch artifact)
  - Entire Leiden cluster 13 (drvi_leiden, res=1.0): 363 cells, 361 of them
    B cells, 83.5% from a single donor (k11) -> suspected batch
    artifact (markers: ABCA6, FCRL2, FCER2, CLNK, TCF4, IGKC — same
    signature found in the B-cell subclustering check, see 2_8_c)
  - PF4+ cells (gene expression, aggressive manual threshold=0.12, instead
    of the data-driven detected valley at 0.95 — 0.12 and 0.2 yield almost
    identical candidate counts (4972 vs. 4966), the gap between them is
    essentially empty -> that is the actual valley floor) -> spread across
    all cell types (no donor dominance), suspected platelet/ambient-RNA
    contamination, see candidate_cells_gene_PF4_in_all.csv.
    Megakaryocytes are excluded from this (genuine, high PF4 expression is
    correct biology for this cell type, not contamination).

Usage:
    conda run -n mapra_cytokines python 2_8_a_drvi_create_annotated_copy.py

In [ ]:
import os
import numpy as np
import scanpy as sc
import pandas as pd

In [ ]:
DRVI_INPUT = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration.h5ad"
OUTPUT_DIR = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation"
OUT_H5AD = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration_curated.h5ad"
CELLTYPE_KEY = "cell_type_Scanorama"

ANNOTATIONS = {
    "candidate_cells_DR9neg.csv": "Tregs",
    "candidate_cells_DR17pos.csv": "pDC",
    "candidate_cells_DR28pos.csv": "Plasma cells",
}
# These CSVs are already filtered to the respective cell type (from
# 2_7_a --celltype-filter or 2_7_c), so a plain union without re-masking by
# cell type is sufficient.
REMOVE_CSVS = {
    "candidate_cells_DR21pos_in_Bcell.csv": "DR21+ in B-cell (B/NK doublets)",
    "candidate_cells_DR35pos_in_MonocytesCD14_Dendritic_MonocytesCD16FCGR3A.csv":
        "DR35+ in Monocytes-CD14/Dendritic/Monocytes-CD16_FCGR3A (B/myeloid doublets)",
    "candidate_cells_gene_IGHG3_in_Bcell.csv": "IGHG3+ in B-cell (k28 mini-cluster)",
    "candidate_cells_gene_PF4_in_all.csv": "PF4+ (threshold=0.12, suspected platelets/ambient RNA)",
}
# Cell types excluded from certain removal reasons because the signal there
# is genuine biology rather than an artifact.
REMOVE_EXCEPTIONS = {
    "candidate_cells_gene_PF4_in_all.csv": ["Megakaryocytes"],
}
LEIDEN_KEY = "drvi_leiden"
REMOVE_LEIDEN_CLUSTERS = ["13"]  # k11-dominated B-cell mini-cluster, see 2_8_c


def load_barcodes(fname: str) -> set:
    path = os.path.join(OUTPUT_DIR, fname)
    return set(pd.read_csv(path)["cell_barcode"])


print("=== Load data ===")
adata = sc.read_h5ad(DRVI_INPUT)
print(f"{adata.n_obs} cells, {adata.n_vars} genes")

### Re-annotation

In [ ]:
print("\n=== Re-annotation ===")
new_celltype = adata.obs[CELLTYPE_KEY].astype(str).copy()

for fname, label in ANNOTATIONS.items():
    barcodes = load_barcodes(fname)
    mask = adata.obs_names.isin(barcodes)
    print(f"{fname} -> '{label}': {mask.sum()} cells "
          f"(before: {new_celltype[mask].value_counts().to_dict()})")
    new_celltype[mask] = label

adata.obs["cell_type_curated"] = pd.Categorical(new_celltype)
print("\ncell_type_curated distribution:")
print(adata.obs["cell_type_curated"].value_counts())

### Removal of suspected doublets/batch artifacts

In [ ]:
print("\n=== Removal ===")

remove_masks = {}
for fname, desc in REMOVE_CSVS.items():
    barcodes = load_barcodes(fname)
    mask = adata.obs_names.isin(barcodes)
    exceptions = REMOVE_EXCEPTIONS.get(fname)
    if exceptions:
        is_exception = adata.obs[CELLTYPE_KEY].isin(exceptions).values
        n_spared = int((mask & is_exception).sum())
        mask = mask & ~is_exception
        print(f"  ({n_spared} cells from {exceptions} excluded from '{desc}')")
    remove_masks[desc] = mask
    print(f"{desc}: {mask.sum()} cells")

leiden_desc = f"{LEIDEN_KEY} in {REMOVE_LEIDEN_CLUSTERS} (k11 mini-cluster)"
leiden_mask = adata.obs[LEIDEN_KEY].astype(str).isin(REMOVE_LEIDEN_CLUSTERS).values
remove_masks[leiden_desc] = leiden_mask
print(f"{leiden_desc}: {leiden_mask.sum()} cells")

remove_mask = np.zeros(adata.n_obs, dtype=bool)
for mask in remove_masks.values():
    remove_mask |= mask

descs = list(remove_masks.keys())
for i in range(len(descs)):
    for j in range(i + 1, len(descs)):
        overlap = int((remove_masks[descs[i]] & remove_masks[descs[j]]).sum())
        if overlap:
            print(f"Overlap '{descs[i]}' & '{descs[j]}': {overlap}")

print(f"Total to remove: {int(remove_mask.sum())} cells "
      f"({100*remove_mask.sum()/adata.n_obs:.2f}% of all cells)")

# Sanity check: does this also remove newly annotated cells (Tregs/pDC/Plasma)?
for fname, label in ANNOTATIONS.items():
    barcodes = load_barcodes(fname)
    mask = adata.obs_names.isin(barcodes)
    overlap = int((mask & remove_mask).sum())
    if overlap:
        print(f"  WARNING: {overlap} '{label}' cells are also on the removal list!")

adata_curated = adata[~remove_mask].copy()
print(f"\nRemaining cells: {adata_curated.n_obs} / {adata.n_obs}")

### Save

In [ ]:
print(f"\n=== Save ===")
adata_curated.write_h5ad(OUT_H5AD)
print(f"Saved: {OUT_H5AD}")
print(f"\ncell_type_curated distribution (final):")
print(adata_curated.obs["cell_type_curated"].value_counts())

## 2. Cell-type UMAP of the curated copy

*Source: `2_8_b_drvi_curated_celltype_umap.py`*

UMAP of the curated cell-type annotation (`cell_type_curated`) from
data_for_practicum_post_integration_curated.h5ad (2_8_a).

Uses the already existing DRVI UMAP from embed.h5ad (reduced to the
remaining cells) — no recomputation needed since only 2% of the cells
were removed.

Usage:
    conda run -n mapra_cytokines python 2_8_b_drvi_curated_celltype_umap.py

In [ ]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
CURATED_H5AD = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration_curated.h5ad"
OUTPUT_DIR = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation"
EMBED_H5AD = os.path.join(OUTPUT_DIR, "embed.h5ad")

print("=== Load data ===")
adata = sc.read_h5ad(CURATED_H5AD)
print(f"{adata.n_obs} cells")

embed = sc.read_h5ad(EMBED_H5AD)
umap_df = pd.DataFrame(embed.obsm["X_umap"], index=embed.obs_names)
adata.obsm["X_umap"] = umap_df.loc[adata.obs_names].values

sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.umap(adata, color="cell_type_curated", ax=ax, show=False,
           title=f"cell_type_curated (n={adata.n_obs} cells)", legend_loc="right margin",
           legend_fontsize=8, size=4)

out = os.path.join(OUTPUT_DIR, "umap_cell_type_curated.png")
plt.savefig(out, bbox_inches="tight", dpi=150)
plt.close("all")
print(f"Saved: {out}")

## 3. B-cell subclustering check (mini-cluster diagnosis)

*Source: `2_8_c_drvi_bcell_subcluster_check.py`*

Checks whether a conspicuous mini-cluster still exists within the (already
cleaned) B cells in the curated dataset: subclustering (Leiden on the DRVI
embedding) restricted to B cells, then donor composition and top marker
genes per cluster to identify small/donor-dominated clusters.

Usage:
    conda run -n mapra_cytokines python 2_8_c_drvi_bcell_subcluster_check.py

In [ ]:
import os
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
CURATED_H5AD = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration_curated.h5ad"
OUTPUT_DIR = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation"
CELLTYPE_KEY = "cell_type_curated"
RESOLUTION = 1.0

print("=== Load data ===")
adata = sc.read_h5ad(CURATED_H5AD)
b = adata[adata.obs[CELLTYPE_KEY] == "B-cell"].copy()
print(f"B cells: {b.n_obs}")

print("\n=== Subclustering (Leiden on X_drvi) ===")
sc.pp.neighbors(b, use_rep="X_drvi", n_neighbors=15)
sc.tl.umap(b, random_state=0)
sc.tl.leiden(b, resolution=RESOLUTION, key_added="b_subcluster")

cluster_sizes = b.obs["b_subcluster"].value_counts().sort_index()
print("\nCluster sizes:")
print(cluster_sizes)

### Donor composition per cluster

In [ ]:
patient_id = b.obs["sample_id"].astype(str).str.split(".", n=1).str[0]
b.obs["patient_id"] = patient_id

print("\n=== Donor dominance per cluster ===")
dominance_rows = []
for cl in cluster_sizes.index:
    sub = patient_id[b.obs["b_subcluster"] == cl]
    top_donor = sub.value_counts().idxmax()
    top_frac = sub.value_counts().iloc[0] / len(sub)
    dominance_rows.append({
        "cluster": cl, "n_cells": len(sub),
        "top_donor": top_donor, "top_donor_fraction": top_frac,
    })
    print(f"  Cluster {cl}: n={len(sub)}, dominant donor={top_donor} ({100*top_frac:.1f}%)")

dominance_df = pd.DataFrame(dominance_rows).sort_values("top_donor_fraction", ascending=False)
dominance_df.to_csv(os.path.join(OUTPUT_DIR, "bcell_subcluster_donor_dominance.csv"), index=False)

# ── Marker genes for the most conspicuous (smallest / most strongly
#    donor-dominated) cluster ─────────────────────────────────────────────────

suspect_cluster = dominance_df.iloc[0]["cluster"]
print(f"\n=== Marker genes for the most conspicuous cluster ({suspect_cluster}) ===")

b.X = b.layers["log1p_norm"]
sc.tl.rank_genes_groups(b, "b_subcluster", groups=[suspect_cluster], reference="rest", method="wilcoxon")
markers = sc.get.rank_genes_groups_df(b, group=suspect_cluster).head(20)
print(markers[["names", "logfoldchanges", "pvals_adj"]].to_string(index=False))
markers.to_csv(os.path.join(OUTPUT_DIR, f"bcell_subcluster_{suspect_cluster}_markers.csv"), index=False)

### Plot: UMAP of the B cells, colored by subcluster and by donor

In [ ]:
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sc.pl.umap(b, color="b_subcluster", ax=axes[0], show=False, title="B-cell subclusters", legend_loc="on data")
top_donors = patient_id.value_counts().head(8).index.tolist()
b.obs["patient_id_top"] = patient_id.where(patient_id.isin(top_donors), "other")
sc.pl.umap(b, color="patient_id_top", ax=axes[1], show=False, title="Donor (top 8 + other)")
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, "bcell_subcluster_umap.png")
plt.savefig(out, bbox_inches="tight", dpi=150)
plt.close("all")
print(f"\nSaved: {out}")
print(f"Saved: bcell_subcluster_donor_dominance.csv")
print(f"Saved: bcell_subcluster_{suspect_cluster}_markers.csv")

## 4. Leiden cluster UMAP of all cells (curated)

*Source: `2_8_e_drvi_all_leiden_umap.py`*

UMAP of the Leiden clusters (drvi_leiden, res=1.0) for ALL cells in the
curated dataset (after removal of the doublets/artifacts).

Usage:
    conda run -n mapra_cytokines python 2_8_e_drvi_all_leiden_umap.py

In [ ]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
CURATED_H5AD = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration_curated.h5ad"
OUTPUT_DIR = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation"
EMBED_H5AD = os.path.join(OUTPUT_DIR, "embed.h5ad")
LEIDEN_KEY = "drvi_leiden"

print("=== Load data ===")
adata = sc.read_h5ad(CURATED_H5AD)
print(f"{adata.n_obs} cells")

embed = sc.read_h5ad(EMBED_H5AD)
umap_df = pd.DataFrame(embed.obsm["X_umap"], index=embed.obs_names)
adata.obsm["X_umap"] = umap_df.loc[adata.obs_names].values

sizes = adata.obs[LEIDEN_KEY].value_counts().sort_index()
adata.obs[f"{LEIDEN_KEY}_label"] = adata.obs[LEIDEN_KEY].map(lambda c: f"{c} (n={sizes[c]})")

sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.umap(adata, color=f"{LEIDEN_KEY}_label", ax=ax, show=False,
           title=f"{LEIDEN_KEY} (res=1.0, n={adata.n_obs} cells)",
           legend_loc="right margin", legend_fontsize=8)

out = os.path.join(OUTPUT_DIR, "umap_all_leiden_curated.png")
plt.savefig(out, bbox_inches="tight", dpi=150)
plt.close("all")
print(f"Saved: {out}")

## 5. Filter active dimensions

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold

In [ ]:
CURATED_H5AD = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration_curated.h5ad"
OUTPUT_DIR = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation"
MAPPING_CSV = os.path.join(OUTPUT_DIR, "factor_title_mapping.csv")
OUT_H5AD = "/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration_curated_activedims_split.h5ad"

# Only these three classification values are mapped to a condition at all
# (rule: only "acs_w_o_infection" counts as ACS, the other ACS subtypes are
# out; "non-CCS" is now only "vollstaendiger_ausschluss", "koronarsklerose"
# is out). All other patients (acs_w_infection, acs_subacute,
# koronarsklerose) are excluded entirely — neither condition, split, nor
# training.
CONDITION_MAP = {
    "acs_w_o_infection": "ACS",
    "ccs": "CCS",
    "vollstaendiger_ausschluss": "non-CCS",
}
SEED = 0

print("=== Load data ===")
adata = sc.read_h5ad(CURATED_H5AD)
print(f"{adata.n_obs} cells, X_drvi shape: {adata.obsm['X_drvi'].shape}")

In [ ]:
dim_map = pd.read_csv(MAPPING_CSV, index_col=0)
dim_map["vanished"] = dim_map["vanished"].astype(bool)

n_dims = adata.obsm["X_drvi"].shape[1]
dim_names = [f"DR{i+1}" for i in range(n_dims)]  # column order of X_drvi
active_dims = [d for d in dim_names if not dim_map.loc[d, "vanished"]]
vanished_dims = [d for d in dim_names if dim_map.loc[d, "vanished"]]
active_idx = [dim_names.index(d) for d in active_dims]

print(f"Active dimensions ({len(active_dims)}/{n_dims}): {active_dims}")
print(f"Removed (vanished) dimensions ({len(vanished_dims)}): {vanished_dims}")

adata.obsm["X_drvi"] = adata.obsm["X_drvi"][:, active_idx]
adata.uns["X_drvi_active_dims"] = active_dims
print(f"New X_drvi shape: {adata.obsm['X_drvi'].shape}")

## 6. Split (stratified: 15% test holdout + 5-fold StratifiedKFold)

Condition mapping restricted: **ACS** = only `acs_w_o_infection`
(19 patients), **CCS** = `ccs` (16), **non-CCS** = only
`vollstaendiger_ausschluss` (10). `acs_w_infection`, `acs_subacute` and
`koronarsklerose` (16 patients in total) are excluded entirely — neither
in condition, split, nor RF training.

Split at the **patient level** (sample_id -> patient, all timepoints stay
together): 15% test holdout (never used for training/CV), the rest via
`StratifiedKFold` in 5 folds. Ends up in `adata.obs["holdout_stratified"]`
("cv"/"test"/"excluded") and `adata.obs["cv_fold_stratified"]` (0-4 in the
CV pool, -1 in the test holdout, -2 excluded).

### Patient table + condition

In [ ]:
patient_id = adata.obs["sample_id"].astype(str).str.split(".", n=1).str[0]
condition = adata.obs["classification"].astype(str).map(CONDITION_MAP)
# Patients with an unmapped classification (acs_w_infection, acs_subacute,
# koronarsklerose) are deliberately NOT dropped here, but get condition=NaN
# and are then explicitly assigned to the "excluded" category below.

patient_table_all = (
    pd.DataFrame({"patient_id": patient_id.values, "condition": condition.values})
    .drop_duplicates("patient_id")
    .reset_index(drop=True)
)
excluded_patients = patient_table_all.loc[patient_table_all["condition"].isna(), "patient_id"].tolist()
patient_table = patient_table_all.dropna(subset=["condition"]).reset_index(drop=True)

print(f"{len(patient_table_all)} patients in total")
print(f"{len(excluded_patients)} excluded (classification outside the mapping)")
print(f"{len(patient_table)} patients in the condition scope:")
print(patient_table["condition"].value_counts())

### Scheme 1: `stratified` (15% holdout + 5-fold StratifiedKFold)

In [ ]:
TEST_FRACTION = 0.15
N_FOLDS = 5

cv_pool, test_pool = train_test_split(
    patient_table, test_size=TEST_FRACTION, stratify=patient_table["condition"],
    random_state=SEED,
)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cv_pool = cv_pool.reset_index(drop=True)
cv_pool["cv_fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(cv_pool["patient_id"], cv_pool["condition"])):
    cv_pool.loc[val_idx, "cv_fold"] = fold

assign = pd.concat([
    test_pool.assign(cv_fold=-1, holdout="test"),
    cv_pool.assign(holdout="cv"),
])[["patient_id", "holdout", "cv_fold"]].set_index("patient_id")

patient_id_to_holdout = assign["holdout"].to_dict()
patient_id_to_fold = assign["cv_fold"].to_dict()
# Excluded patients: holdout="excluded", cv_fold=-2 (to distinguish from
# -1="test holdout" within the condition scope).
for p in excluded_patients:
    patient_id_to_holdout[p] = "excluded"
    patient_id_to_fold[p] = -2

adata.obs["holdout_stratified"] = pd.Categorical(
    patient_id.map(patient_id_to_holdout).values, categories=["cv", "test", "excluded"]
)
adata.obs["cv_fold_stratified"] = patient_id.map(patient_id_to_fold).values.astype(int)

print(f"Test holdout: {len(test_pool)} patients, CV pool: {len(cv_pool)} patients, "
      f"Excluded: {len(excluded_patients)} patients")
print(pd.crosstab(cv_pool["condition"], cv_pool["cv_fold"]))

### Overview + sanity checks

In [ ]:
h = adata.obs["holdout_stratified"]
n_test_patients = patient_id[h.values == "test"].nunique()
n_cv_patients = patient_id[h.values == "cv"].nunique()
n_excl_patients = patient_id[h.values == "excluded"].nunique()
print(f"stratified: {n_cv_patients} in the CV pool, {n_test_patients} in the holdout, "
      f"{n_excl_patients} excluded")

check = pd.DataFrame({
    "patient_id": patient_id.values,
    "holdout": adata.obs["holdout_stratified"].values,
}).drop_duplicates("patient_id")
dup = check.groupby("patient_id").size()
assert (dup == 1).all(), "Patient assigned multiple times!"
print("Sanity check OK: every patient is assigned unambiguously.")

## 7. Split for timepoint prediction (ACS_sterile only, patient-grouped)

A different question than Section 6: here the **timepoint** (TP1-TP4)
should be predicted, not the condition. Only **ACS_sterile**
(`classification == "acs_w_o_infection"`, 19 patients, 62 samples)
is used — all other cells get `holdout_timepoint="excluded"`.

Important difference from Section 6: there, values were averaged per
patient across *all* timepoints (pseudobulk at the patient level), because
the timepoint was irrelevant there. Here the timepoint is exactly the
target variable — so the **sample level** (patient.timepoint, e.g. "m6.1")
is preserved, not the patient level.

Still, the same rule applies: all samples from the **same patient** (e.g.
m6.1, m6.2, m6.3, m6.4) must end up in the **same** split (train/val/test or
CV fold), otherwise patient identity leaks between the sets. This is solved
via `StratifiedGroupKFold` — `groups=patient_id` enforces that patients stay
together, `y=timepoint` keeps the timepoint distribution as balanced as
possible across the folds/splits.

Procedure (two-stage, as in Section 6):
1. `StratifiedGroupKFold(n_splits=5)` across all 62 samples (grouped by
   patient, stratified by timepoint) — the first fold (~20%, all timepoints
   of the respective patients) becomes the test holdout.
2. `StratifiedGroupKFold(n_splits=5)` again on the remaining samples for
   the actual 5-fold CV.

Columns: `adata.obs["holdout_timepoint"]` ("cv"/"test"/"excluded") and
`adata.obs["cv_fold_timepoint"]` (0-4 in the CV pool, -1 in the test
holdout, -2 excluded).

### Sample table (ACS_sterile only)

In [ ]:
ACS_STERILE_VALUE = "acs_w_o_infection"
TIMEPOINT_N_FOLDS = 5
TIMEPOINT_TEST_N_SPLITS = 5  # 1/5 ~ 20% of patients -> test holdout

is_acs_sterile = (adata.obs["classification"].astype(str) == ACS_STERILE_VALUE).values
sample_id_full = adata.obs["sample_id"].astype(str)
patient_id_full = sample_id_full.str.split(".", n=1).str[0]
timepoint_full = sample_id_full.str.split(".", n=1).str[1]

sample_table = (
    pd.DataFrame({
        "sample_id": sample_id_full.values,
        "patient_id": patient_id_full.values,
        "timepoint": timepoint_full.values,
        "is_acs_sterile": is_acs_sterile,
    })
    .drop_duplicates("sample_id")
    .reset_index(drop=True)
)
sample_table = sample_table[sample_table["is_acs_sterile"]].reset_index(drop=True)

print(f"{sample_table['patient_id'].nunique()} patients, {len(sample_table)} samples (ACS_sterile)")
print(sample_table["timepoint"].value_counts().sort_index())

### Two-stage, patient-grouped split

In [ ]:
# Stage 1: test holdout (~20% of patients, all their timepoints together)
sgkf_test = StratifiedGroupKFold(n_splits=TIMEPOINT_TEST_N_SPLITS, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf_test.split(
    sample_table, sample_table["timepoint"], groups=sample_table["patient_id"]
))
cv_pool_samples = sample_table.iloc[train_idx].reset_index(drop=True)
test_samples = sample_table.iloc[test_idx].reset_index(drop=True)

print(f"Test holdout: {test_samples['patient_id'].nunique()} patients, {len(test_samples)} samples")
print(f"CV pool: {cv_pool_samples['patient_id'].nunique()} patients, {len(cv_pool_samples)} samples")

# Stage 2: 5-fold CV on the rest, again patient-grouped
sgkf_cv = StratifiedGroupKFold(n_splits=TIMEPOINT_N_FOLDS, shuffle=True, random_state=SEED)
cv_pool_samples["cv_fold"] = -1
for fold, (_, val_idx) in enumerate(sgkf_cv.split(
    cv_pool_samples, cv_pool_samples["timepoint"], groups=cv_pool_samples["patient_id"]
)):
    cv_pool_samples.loc[val_idx, "cv_fold"] = fold

print("\nSamples per fold x timepoint (CV pool):")
print(pd.crosstab(cv_pool_samples["timepoint"], cv_pool_samples["cv_fold"]))

# Sanity check: no patient spans multiple folds
patient_folds = cv_pool_samples.groupby("patient_id")["cv_fold"].nunique()
assert (patient_folds == 1).all(), "Patient spread across multiple CV folds!"
print("Sanity check OK: every patient stays within a single CV fold.")

### Assign columns

In [ ]:
sample_to_holdout = {sid: "test" for sid in test_samples["sample_id"]}
sample_to_holdout.update({sid: "cv" for sid in cv_pool_samples["sample_id"]})
sample_to_fold = {sid: -1 for sid in test_samples["sample_id"]}
sample_to_fold.update(dict(zip(cv_pool_samples["sample_id"], cv_pool_samples["cv_fold"])))

# All samples outside ACS_sterile: "excluded"/-2
holdout_timepoint = sample_id_full.map(sample_to_holdout).fillna("excluded")
cv_fold_timepoint = sample_id_full.map(sample_to_fold).fillna(-2)

adata.obs["holdout_timepoint"] = pd.Categorical(
    holdout_timepoint.values, categories=["cv", "test", "excluded"]
)
adata.obs["cv_fold_timepoint"] = cv_fold_timepoint.values.astype(int)

print(adata.obs["holdout_timepoint"].value_counts())

### Save

In [ ]:
adata.write_h5ad(OUT_H5AD)
print(f"Saved: {OUT_H5AD}")

## 8. Plasma cell counts in the RF classifier groups

Sanity check: how many Plasma cells (`cell_type_curated == "Plasma cells"`) end up in each group actually used for training/evaluation — the condition groups (ACS/CCS/non-CCS) from `rf_condition_classifier.py` (`holdout_stratified`/`cv_fold_stratified`, patient-level pseudobulk) and the timepoint groups (TP1-TP4) from `rf_timepoint_classifier.py` (`holdout_timepoint`/`cv_fold_timepoint`, sample-level pseudobulk). Patients/samples marked `"excluded"` are outside the respective RF scope and are left out here.

In [ ]:
plasma_mask = (adata.obs["cell_type_curated"] == "Plasma cells").values
print(f"Total Plasma cells: {int(plasma_mask.sum())} / {adata.n_obs} cells")

# ── RF condition classifier (schema: stratified, patient-level pseudobulk) ──
print("\n=== Plasma cells in the condition-classifier groups (ACS/CCS/non-CCS) ===")
holdout_cond = adata.obs["holdout_stratified"].astype(str).values

cond_df = pd.DataFrame({
    "patient_id": patient_id.values,
    "condition": condition.values,
    "holdout": holdout_cond,
    "is_plasma": plasma_mask,
})
cond_in_scope = cond_df[cond_df["holdout"] != "excluded"]

print("Plasma cells per condition (CV-pool + test holdout only):")
print(cond_in_scope.loc[cond_in_scope["is_plasma"]].groupby("condition", observed=True).size())

print("\nPlasma cells per condition x holdout (cv vs. test):")
print(pd.crosstab(cond_in_scope.loc[cond_in_scope["is_plasma"], "condition"],
                   cond_in_scope.loc[cond_in_scope["is_plasma"], "holdout"]))

n_patients_with_plasma = (
    cond_in_scope[cond_in_scope["is_plasma"]]
    .drop_duplicates("patient_id")
    .groupby("condition", observed=True).size()
)
n_patients_total = cond_in_scope.drop_duplicates("patient_id").groupby("condition", observed=True).size()
print("\nPatients with >=1 Plasma cell, per condition (of total patients in that condition):")
print(pd.DataFrame({"patients_with_plasma": n_patients_with_plasma,
                     "patients_total": n_patients_total}))

# ── RF timepoint classifier (ACS_sterile only, sample-level pseudobulk) ─────
print("\n=== Plasma cells in the timepoint-classifier groups (TP1-TP4) ===")
timepoint_split = sample_id_full.str.split(".", n=1)
timepoint_all = np.where(timepoint_split.str[1].notna(), "TP" + timepoint_split.str[1], "control")
holdout_tp = adata.obs["holdout_timepoint"].astype(str).values

tp_df = pd.DataFrame({
    "sample_id": sample_id_full.values,
    "timepoint": timepoint_all,
    "holdout": holdout_tp,
    "is_plasma": plasma_mask,
})
tp_in_scope = tp_df[tp_df["holdout"] != "excluded"]

print("Plasma cells per timepoint (CV-pool + test holdout only):")
print(tp_in_scope.loc[tp_in_scope["is_plasma"]].groupby("timepoint", observed=True).size())

print("\nPlasma cells per timepoint x holdout (cv vs. test):")
print(pd.crosstab(tp_in_scope.loc[tp_in_scope["is_plasma"], "timepoint"],
                   tp_in_scope.loc[tp_in_scope["is_plasma"], "holdout"]))

n_samples_with_plasma = (
    tp_in_scope[tp_in_scope["is_plasma"]]
    .drop_duplicates("sample_id")
    .groupby("timepoint", observed=True).size()
)
n_samples_total = tp_in_scope.drop_duplicates("sample_id").groupby("timepoint", observed=True).size()
print("\nSamples with >=1 Plasma cell, per timepoint (of total samples in that timepoint):")
print(pd.DataFrame({"samples_with_plasma": n_samples_with_plasma,
                     "samples_total": n_samples_total}))

## 9. Plasma cells within ACS, broken down by infection status

The `classification` column has three ACS subtypes: `acs_w_o_infection` (= "ACS" in the RF condition mapping), `acs_w_infection`, and `acs_subacute` (the latter two are excluded from the RF condition classifier, see Section 6). Here we look at **all** ACS patients regardless of RF scope and check whether the Plasma cell population (`cell_type_curated == "Plasma cells"`) is concentrated in the infection patients (`acs_w_infection`) rather than spread evenly — which would suggest the DR28+ Plasma cell signal is (partly) an infection-driven plasmablast response rather than a general ACS marker.

In [ ]:
acs_classes = ["acs_w_o_infection", "acs_w_infection", "acs_subacute"]
classification_all = adata.obs["classification"].astype(str)
acs_mask = classification_all.isin(acs_classes).values
print(f"ACS cells (all subtypes, all timepoints): {int(acs_mask.sum())} / {adata.n_obs}")

acs_df = pd.DataFrame({
    "patient_id": patient_id.values,
    "classification": classification_all.values,
    "is_plasma": plasma_mask,
})[acs_mask]

print("\nCells per ACS subtype (infection status):")
print(acs_df["classification"].value_counts())

# ── Cell-level: Plasma-cell share per subtype ────────────────────────────────
plasma_counts = acs_df[acs_df["is_plasma"]].groupby("classification", observed=True).size()
total_counts = acs_df.groupby("classification", observed=True).size()
cell_summary = pd.DataFrame({
    "plasma_cells": plasma_counts,
    "total_cells": total_counts,
}).fillna(0).astype({"plasma_cells": int, "total_cells": int})
cell_summary["pct_plasma"] = 100 * cell_summary["plasma_cells"] / cell_summary["total_cells"]
cell_summary = cell_summary.reindex(acs_classes)

print("\nPlasma cells per ACS subtype (cell level):")
print(cell_summary)

# ── Patient-level: how many patients per subtype actually contribute ────────
patient_summary = (
    acs_df.groupby(["classification", "patient_id"], observed=True)["is_plasma"]
    .agg(n_plasma="sum", n_cells="count")
    .reset_index()
)
patient_summary["pct_plasma"] = 100 * patient_summary["n_plasma"] / patient_summary["n_cells"]

print("\nPer-patient Plasma cell counts within ACS (sorted by n_plasma, top 15):")
print(patient_summary.sort_values("n_plasma", ascending=False).head(15).to_string(index=False))

n_patients_with_plasma = (
    patient_summary[patient_summary["n_plasma"] > 0]
    .groupby("classification", observed=True)["patient_id"].nunique()
)
n_patients_total = patient_summary.groupby("classification", observed=True)["patient_id"].nunique()
patient_level_summary = pd.DataFrame({
    "patients_with_plasma": n_patients_with_plasma,
    "patients_total": n_patients_total,
}).fillna(0).astype(int).reindex(acs_classes)
print("\nPatients with >=1 Plasma cell, per ACS subtype (of total ACS patients in that subtype):")
print(patient_level_summary)

# ── Plot: Plasma-cell share per ACS subtype ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(cell_summary.index, cell_summary["pct_plasma"], color="#8172B2")
for i, (pct, n_plasma, n_total) in enumerate(zip(cell_summary["pct_plasma"],
                                                   cell_summary["plasma_cells"],
                                                   cell_summary["total_cells"])):
    ax.text(i, pct, f"{n_plasma}/{n_total}\n({pct:.2f}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("% Plasma cells")
ax.set_title("Plasma cell share within ACS, by infection status")
ax.set_ylim(0, cell_summary["pct_plasma"].max() * 1.35)
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, "plasma_cells_by_acs_infection_status.png")
plt.savefig(out, bbox_inches="tight", dpi=150)
plt.close("all")
print(f"\nSaved: {out}")